# ACE-Net — Stage-2 (fixed split) + Evaluation on Colab

Runs on a Colab **T4 GPU** (16 GB) — far more headroom than the local 6 GB box,
so no OOM and Stage-2 finishes in minutes.

**What this does**
1. Clone the baseline repo (code) from GitHub
2. Install deps
3. Mount Google Drive and unzip the CREMA-D vectors (~3.4 GB) into `data/`
4. Upload the trained Stage-1 checkpoints
5. Retrain **Stage-2** with the group-aware (leakage-free) split
6. Evaluate Tables 2/3/4

**Before running, prepare on your machine:**
- Zip the 4 CREMA dirs:
  `data/CREMA-D/{GENUINE_LastHalf,GENUINE_FirstHalf,FAKE_Paradigm1,FAKE_Paradigm2}`
  into `cremad_vectors.zip` and upload it to your Google Drive root.
- Have the 4 Stage-1 checkpoints ready to upload:
  `stage1_visual_crema.pt`, `stage1_speech_text_crema.pt`
  (MELD ckpts only needed if you also want MELD emotion eval).

> Runtime → Change runtime type → **T4 GPU** before running.

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Clone repo

In [ ]:
%cd /content
!rm -rf Baseline_Training
!git clone https://github.com/gjvlio/Baseline_Training.git
%cd Baseline_Training
!git log --oneline -1

## 3. Install dependencies

In [ ]:
!pip -q install torch torchvision torchaudio transformers librosa pillow
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())

## 4. Get the CREMA-D vectors

Mount Drive and unzip `cremad_vectors.zip` into `data/CREMA-D/`. The zip should
contain the four dirs at its top level (GENUINE_LastHalf, GENUINE_FirstHalf,
FAKE_Paradigm1, FAKE_Paradigm2).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile
ZIP = '/content/drive/MyDrive/cremad_vectors.zip'   # adjust path if needed
DST = '/content/Baseline_Training/data/CREMA-D'
os.makedirs(DST, exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(DST)
print("unzipped. contents:")
print(os.listdir(DST))

## 5. Upload Stage-1 checkpoints

Upload the trained `.pt` files into `checkpoints/`. (CREMA visual + speech are
required for Stage-2; add MELD ckpts if you also want MELD emotion eval.)

In [ ]:
import os
os.makedirs('checkpoints', exist_ok=True)
from google.colab import files
print("select stage1_visual_crema.pt, stage1_speech_text_crema.pt (+ MELD if wanted)")
up = files.upload()
for name in up:
    os.replace(name, f'checkpoints/{name}')
print(os.listdir('checkpoints'))

## 6. Sanity: data resolves + leakage-free split

Confirms the vectors load and the group-aware Stage-2 split has zero
actor overlap between train and test.

In [ ]:
import sys; sys.path.insert(0, '/content/Baseline_Training')
from src.data import manifests
from src.train_utils import group_aware_split
from src.config import TrainConfig
import random
cfg = TrainConfig()
g, f = manifests.build_stage2_samples()
print("genuine:", len(g), "fake:", len(f))

# replicate balanced build to check split
rng = random.Random(cfg.seed)
p1=[s for s in f if s.emotion=='crema_fake_p1']; p2=[s for s in f if s.emotion=='crema_fake_p2']
ne=min(len(p1),len(p2)); rng.shuffle(p1); rng.shuffle(p2)
fakes=p1[:ne]+p2[:ne]; rng.shuffle(fakes)
n=min(len(g),len(fakes)); rng.shuffle(g); chosen=g[:n]+fakes[:n]
tr,va,te = group_aware_split(chosen, lambda s:s.group_key, (0.8,0.1,0.1), cfg.seed)
trg={s.group_key for s in tr}; teg={s.group_key for s in te}
print(f"split train/val/test = {len(tr)}/{len(va)}/{len(te)}")
print("actor-group overlap train&test:", len(trg & teg), "(must be 0)")

## 7. Train Stage-2 (leakage-free split)

T4 is fast — run more epochs than the local time-capped run for better
convergence. No `--max-minutes` needed here.

In [ ]:
!cd /content/Baseline_Training && PYTHONPATH=. python -m src.train_stage2 \
    --batch-size 32 --epochs 30 --early-stop 8 --num-workers 2

## 8. Evaluate Table 4 — forgery detection by type

In [ ]:
!cd /content/Baseline_Training && PYTHONPATH=. python -m src.eval_stage2

## 9. (Optional) Tables 2/3 — emotion recognition

Needs the corresponding Stage-1 checkpoints uploaded in step 5. MELD also needs
the MELD vectors unzipped (not included in the CREMA-only zip).

In [ ]:
!cd /content/Baseline_Training && PYTHONPATH=. python -m src.eval_stage1 --branch visual      --dataset crema
!cd /content/Baseline_Training && PYTHONPATH=. python -m src.eval_stage1 --branch speech_text --dataset crema

## 10. Visualize (the results notebook)

Run the repo's `notebooks/results.ipynb` cells, or just read the printed tables
above. The honest Table-4 AUC from step 8 is the headline number — expect it to
be **lower than the leaky 0.998**, which is the fix working.